# Étude comparative des 16 configurations de SloshingPINN

Notebook de protocole et d'analyse pour les quatre options :
- Fourier Features
- Adaptive Sampling
- Temporal Segmentation
- Adaptive Weighting (GradNorm)

Comparer les configurations sur un **jeu de validation commun**, indépendant des données d'entraînement. Les cellules dépendant du format exact des checkpoints et des exports OpenFOAM sont laissées à adapter.


## Imports et paramètres

In [1]:
from pathlib import Path
import sys
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))
print('Répertoire du projet :', ROOT)

from sloshing_pinn_analysis import (
    plot_model_vs_openfoam, plot_model_residual_grids,
    compute_error_metrics, plot_metric_comparison,
)

from visualization import load_model, plot_fields, compute_error_metrics, plot_metric_comparison

SEED = 1234
rng = np.random.default_rng(SEED)

import matplotlib
matplotlib.use("Agg")

def load_model_from_name(config_name, device="cuda"):
    checkpoint_path = ROOT / "models" / config_name / "checkpoint_final.pt"
    config_file = ROOT / "models" / config_name / "config.json"
    return load_model(checkpoint_path, config_file, device)

FIGURES = ROOT / "reports" / "figures"
TABLES = ROOT / "reports" / "tables"
if not FIGURES.exists():
    FIGURES.mkdir()
if not TABLES.exists():
    TABLES.mkdir()

Répertoire du projet : /home/ludodo/SINN


## Plan factoriel des 16 configurations

In [2]:
FEATURES = [
    "Fourier Features",
    "Adaptive Sampling",
    "Temporal Segmentation",
    "Adaptive Weighting",
]

configs_df = pd.DataFrame([
    {"configuration": "model_ff" + str(int(bits[0])) + "_as" + str(int(bits[1])) + "_ts" + str(int(bits[2])) + "_alw" + str(int(bits[3])),
     **dict(zip(FEATURES, bits))}
    for bits in itertools.product([False, True], repeat=4)
])
configs_df.to_csv(ROOT / "reports" / "tables" / "configuration.csv", )
configs_df


,configuration,Fourier Features,Adaptive Sampling,Temporal Segmentation,Adaptive Weighting
0,model_ff0_as0_ts0_alw0,False,False,False,False
1,model_ff0_as0_ts0_alw1,False,False,False,True
2,model_ff0_as0_ts1_alw0,False,False,True,False
3,model_ff0_as0_ts1_alw1,False,False,True,True
4,model_ff0_as1_ts0_alw0,False,True,False,False
5,model_ff0_as1_ts0_alw1,False,True,False,True
6,model_ff0_as1_ts1_alw0,False,True,True,False
7,model_ff0_as1_ts1_alw1,False,True,True,True
8,model_ff1_as0_ts0_alw0,True,False,False,False
9,model_ff1_as0_ts0_alw1,True,False,False,True


## Chargement des données et des modèles

In [3]:
OPENFOAM_FILE = ROOT / "data/sloshing_data.npz"

openfoam_data = np.load(OPENFOAM_FILE, allow_pickle=True)

In [4]:
models = {
    config_row["configuration"]: load_model_from_name(config_row["configuration"])
    for _, config_row in configs_df.iterrows()
}
model = models["model_ff1_as1_ts1_alw1"]  # Example: load the model with all features enabled

In [5]:
list(openfoam_data.keys())

['time',
 'X',
 'u',
 'v',
 'w',
 'alpha',
 'p',
 'p_rgh',
 'k',
 'epsilon',
 'Fx',
 'Fy',
 'Mz']

## Jeu d'évaluation commun

## Comparaison visuelle avec OpenFOAM

In [ ]:
FIELDS_TO_COMPARE = ["u", "v", "alpha"]
TIME_INDICES_TO_PLOT = np.arange(0, 10, 2, dtype=float)

for model_name, model in models.items():
    plot_fields(
        model,
        fields=FIELDS_TO_COMPARE,
        residuals=False,
        openfoam_data=openfoam_data,
        t=TIME_INDICES_TO_PLOT,
        save_path=FIGURES / f"openfoam_vs_{model_name}.png" 
        )



Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.


/home/ludodo/SINN/src/visualization.py:290: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(n, 3, figsize=(15, 5*n))


Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for display: [0.2 2.  4.  6.  8. ] instead of provided.
residuals display disabled when comparing with OpenFOAM data.
Using OpenFOAM times for di

: 

## Cartographie des résidus physiques

In [ ]:
TIMES_FOR_RESIDUALS = np.arange(0, 10, 2, dtype=float)
FIELDS_TO_COMPARE = ["u", "v", "p", "alpha"]
for model_name, model in models.items():
    plot_fields(
        model,
        fields=FIELDS_TO_COMPARE,
        residuals=True,
        t=TIMES_FOR_RESIDUALS,
        save_path=ROOT / "reports"  / "figures" / f"residuals_{model_name}.png")


p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, skipping residuals display.
p not found in residuals, sk

## Évaluation quantitative

Indicateurs 
- MAE
- RMSE
- erreur relative L2
- percentile 95 de l'erreur absolue.

L'alignement point à point entre OpenFOAM et PINN doit être vérifié.

In [ ]:
grid_interval = 1  # Use every 10th point in the spatial grid
grid_interval = 1  # Use every 10th point in the spatial grid
t_interval = 1  # Use every 10th time step

x = openfoam_data["X"][0, ::grid_interval, 0]
y = openfoam_data["X"][0, ::grid_interval, 1]
t = openfoam_data["time"][::t_interval]  # Use every 10th time
openfoam_ref = {
    "u": openfoam_data["u"][::t_interval, ::grid_interval],
    "v": openfoam_data["v"][::t_interval, ::grid_interval],
    "alpha": openfoam_data["alpha"][::t_interval, ::grid_interval],
    "p": openfoam_data["p"][::t_interval, ::grid_interval],
}

N, T = len(x), len(t)

coords = np.empty((T * N, 3), dtype=np.float32)
coords[:, 0] = np.tile(x, T)      # x_i répété pour chaque t
coords[:, 1] = np.tile(y, T)      # y_i répété pour chaque t
coords[:, 2] = np.repeat(t, N)    # t_l répété N fois


In [ ]:
metric_rows = []

import time

for model_name, model in models.items():
    model.eval()  # Set the model to evaluation mode
    # print(f"Computing predictions for model: {model_name}")
    # start_time = time.time()
    preds = model.predict_physical(coords_si=coords) # (T·N, 1)
    predictions = {k: v.reshape(T, N).cpu().detach().numpy() for k, v in preds.items()}    # (T, N)
    # print(f"Predictions computed for model: {model_name} in {time.time() - start_time} s")
    for field in FIELDS_TO_COMPARE:
        ref_values = openfoam_ref[field]   # référence OpenFOAM sur le jeu commun
        pred_values = predictions[field]  # prediction PINN aux mêmes coordonnées/temps
        # print(f"ref_values shape: {ref_values.shape}, pred_values shape: {pred_values.shape}")
        metric_rows.append({
            "configuration": model_name,
            "field": field,
            **compute_error_metrics(ref_values, pred_values),
        })
metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(TABLES / "metrics.csv")
metrics_df

## Comparaison et classement

In [ ]:
alpha_metrics = metrics_df[metrics_df["field"] == "alpha"]
plot_metric_comparison(alpha_metrics, metric="RMSE")
plot_metric_comparison(alpha_metrics, metric="Relative_L2")

for field in ["u", "v", "p", "alpha"]:
    field_metrics = metrics_df[metrics_df["field"] == field]
    for metric in ["RMSE", "Relative_L2", "MAE", "P95_abs_error"]:
        plot_metric_comparison(field_metrics, metric=metric, save_path=FIGURES / f"{field}_{metric}_score.png")


## Effets des options (analyse factorielle descriptive)

Un effet négatif sur RMSE signifie que l'option activée réduit le score moyen. Pour interpréter solidement les effets et interactions, entraînez si possible plusieurs graines par configuration.

In [ ]:
results = pd.merge(configs_df, metrics_df, on="configuration")
results.to_csv(TABLES / "results.csv")

In [ ]:
def summarize_factor_effects(results, score="RMSE", factors=FEATURES, fields=["u", "v", "p", "alpha"]):
    effects_on_fields = []
    for field in fields:
        field_df = results.loc[results["field"] == field]
        rows = []
        for factor in factors:
            mean_on = field_df.loc[field_df[factor] == True, score].mean()
            mean_off = field_df.loc[field_df[factor] == False, score].mean()
            rows.append({
                "field": field,
                "factor": factor,
                "mean_when_on": mean_on,
                "mean_when_off": mean_off,
                "effect_on_minus_off": mean_on - mean_off,
            })
        effects_on_fields.append(pd.DataFrame(rows).sort_values("effect_on_minus_off"))
    return pd.concat(effects_on_fields, ignore_index=True, axis=0)

effects = summarize_factor_effects(results)
effects.to_csv(TABLES / "factorial_analysis.csv")
effects


## Grille d'interprétation et bilan

| Observation | Diagnostic à examiner | Action expérimentale |
|---|---|---|
| Erreur OpenFOAM élevée, résidus PDE faibles | BC/IC, normalisation, champs/units ou validation | Vérifier les conditions et l'alignement des données |
| Erreur et résidus élevés | Optimisation ou capacité du modèle | Revoir architecture, pertes et durée d'entraînement |
| Erreur forte sur l'interface alpha | Interface mal résolue | Raffiner le sampling près de l'interface, revoir VOF |
| Erreurs surtout aux instants rapides | Couverture temporelle insuffisante | Tester segmentation et sampling temporel |
| Gains seulement sur train | Surapprentissage / biais de sampling | Utiliser validation indépendante et commune |
| Certains champs s'améliorent, d'autres se dégradent | Compromis multiobjectif | Analyse Pareto plutôt qu'un score composite arbitraire |

**Bilan à documenter :** meilleur modèle global et par champ, compromis erreur/résidus, stabilité temporelle, dispersion entre graines, et coûts d'entraînement/inférence.
